In [2]:
from obc_syn import synthesize_general, validate_synthesis
from typing import Callable
from tqdm.notebook import tqdm
import numpy as np

In [3]:
OR_FN = lambda x, y, z: (x | y) == z
blocked_sols = []

while len(blocked_sols) < 4:
    sol = synthesize_general(
        OR_FN,
        n_io_var=3,
        n_aux_osc=0,
        block_solution=blocked_sols,
    )
    if sol is None:
        break
    # print("Found solution:\n", sol)
    validate_synthesis(sol, OR_FN, n_io_var=3, n_aux_osc=0)
    blocked_sols.append(sol)

Normalized Coupling matrix:
[[ 0.          0.88888889 -1.77777778  0.88888889]
 [ 0.88888889  0.         -1.77777778  0.88888889]
 [-1.77777778 -1.77777778  0.         -1.77777778]
 [ 0.88888889  0.88888889 -1.77777778  0.        ]]
Truth table and energies:
I/O                 E         
[0, 0, 0]           [-6.0]    
[0, 0, 1]           [2.0]     
[0, 1, 0]           [2.0]     
[0, 1, 1]           [-6.0]    
[1, 0, 0]           [2.0]     
[1, 0, 1]           [-6.0]    
[1, 1, 0]           [18.0]    
[1, 1, 1]           [-6.0]    
Normalized Coupling matrix:
[[ 0.          1.33333333 -2.          0.66666667]
 [ 1.33333333  0.         -2.          0.66666667]
 [-2.         -2.          0.         -1.33333333]
 [ 0.66666667  0.66666667 -1.33333333  0.        ]]
Truth table and energies:
I/O                 E         
[0, 0, 0]           [-8.0]    
[0, 0, 1]           [8.0]     
[0, 1, 0]           [0.0]     
[0, 1, 1]           [-8.0]    
[1, 0, 0]           [0.0]     
[1, 0, 1]        

Examine the scaling of the solution space. Number of parameters scales with N ** 2 while number of ground states scales 2 ** N.

Check if more auxiliary oscillator needed when N increases.

In [4]:
def enumerate_logic_fn(n: int) -> list[Callable]:
    """Return all possible (n-1)-input 1-output logic functions.

    Args:
        n (int): Number of inputs.
    Returns:
        2 ** (2 ** (n-1)) logic functions.
    """

    from itertools import product

    fn_list = []
    truth_table_str_list = []
    for truth_table in product([False, True], repeat=2 ** (n - 1)):

        def fn(*args, table=truth_table):
            input_args = args[:-1]
            index = sum((1 << i) if arg else 0 for i, arg in enumerate(input_args))
            return table[index] == args[-1]

        truth_table_str = "".join(["1" if v else "0" for v in truth_table])
        truth_table_str_list.append(truth_table_str)
        fn_list.append(fn)
    return fn_list, truth_table_str_list


n_osc_needed_overall = {}
for n_input_var in range(2, 5):
    n_io_var = n_input_var + 1
    fn_list, truth_table_str_list = enumerate_logic_fn(n_io_var)
    print(f"n_input_var={n_input_var}, number of functions: {len(fn_list)}")
    if n_input_var >= 3:
        # For large n_input_var, only test a subset of functions
        sample_idxs = np.random.choice(len(fn_list), size=16, replace=False)
        fn_list = [fn_list[i] for i in sample_idxs]
        truth_table_str_list = [truth_table_str_list[i] for i in sample_idxs]
    n_osc_needed_list = []
    for fn in tqdm(fn_list):
        n_aux_osc = 0
        while True:
            sol = synthesize_general(
                fn,
                n_io_var=n_io_var,
                n_aux_osc=n_aux_osc,
                block_solution=[],
            )
            if sol is not None:
                break
            n_aux_osc += 1
        n_osc_needed_list.append(n_io_var + n_aux_osc)

    print("Truth tables and number of additional oscillators needed:")
    for truth_table_str, n_osc_needed in zip(truth_table_str_list, n_osc_needed_list):
        print(f"{truth_table_str}: {n_osc_needed - n_io_var}")

    n_osc_needed_overall[n_input_var] = n_osc_needed_list

n_input_var=2, number of functions: 16


  0%|          | 0/16 [00:00<?, ?it/s]

Truth tables and number of additional oscillators needed:
0000: 0
0001: 0
0010: 0
0011: 0
0100: 0
0101: 0
0110: 1
0111: 0
1000: 0
1001: 1
1010: 0
1011: 0
1100: 0
1101: 0
1110: 0
1111: 0
n_input_var=3, number of functions: 256


  0%|          | 0/16 [00:00<?, ?it/s]

Truth tables and number of additional oscillators needed:
11010101: 1
00010101: 1
00010110: 1
11000010: 1
11001000: 1
00111010: 1
01001000: 1
01001110: 1
00001111: 0
01010111: 1
11101010: 1
10110101: 1
01001101: 0
10111110: 1
10000011: 1
10100001: 1
n_input_var=4, number of functions: 65536


  0%|          | 0/16 [00:00<?, ?it/s]

Truth tables and number of additional oscillators needed:
0000110111101000: 2
1011100001001011: 2
0111110110011000: 2
1110110110111100: 2
0001100111101101: 2
0001001010100001: 2
1100111111010100: 2
1101010110000110: 2
0000101001010001: 2
0001111100000111: 1
1100101000100100: 2
0100110101000101: 1
1000110111000111: 2
1001111000001100: 2
1001101001101010: 2
1001101100101101: 2
